# Section 1 · Your First AI Agent

In this notebook you build your first AI agent with the [Strands Agents SDK](https://strandsagents.com), powered by Amazon Bedrock.

You will:
1. Install the SDK
2. Pick a Bedrock model
3. Create an agent in one line
4. Give it your own tool with `@tool`
5. Give it a prebuilt tool
6. Shape its behaviour with a system prompt

**Before you start**
- AWS credentials configured (`aws configure`, AWS SSO, or a SageMaker execution role)
- Region set to one where Amazon Bedrock is available (this course uses `us-east-1`)

## Step 1: Install dependencies

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Step 2: Choose the model

Every example in this course reads the model from one variable. Change it here (or set the `BEDROCK_MODEL_ID` environment variable) to try a different model.

- `global.anthropic.claude-sonnet-4-6` — default, strong reasoning and tool use
- `global.anthropic.claude-haiku-4-5-20251001-v1:0` — faster and cheaper

In [2]:
import os

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

print(f"Model : {MODEL_ID}")
print(f"Region: {REGION}")

Model : global.anthropic.claude-sonnet-4-6
Region: us-west-2


## Step 3: Create your first agent

In [3]:
from strands import Agent
from strands.models.bedrock import BedrockModel

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

# An agent = a model + (optional) tools + (optional) system prompt
agent = Agent(model=model)

response = agent("Explain Amazon Bedrock AgentCore in three sentences.")

**Amazon Bedrock AgentCore** is a fully managed infrastructure platform from AWS designed to help developers deploy, run, and scale AI agents in production environments. It provides essential runtime capabilities including secure code execution (via a sandboxed environment), memory management, tool/API integration, and an observability layer to monitor agent behavior and performance. By abstracting away the underlying infrastructure complexity, AgentCore allows developers to focus on building agent logic while AWS handles the scalability, security, and reliability of the agent's operating environment.

The agent streams its answer as it generates it. The returned object holds the full result — the final text, token usage, and the stop reason:

In [4]:
print("Stop reason:", response.stop_reason)
print("Tokens used:", response.metrics.accumulated_usage)

Stop reason: end_turn
Tokens used: {'inputTokens': 20, 'outputTokens': 111, 'totalTokens': 131}


## Step 4: Give the agent a tool

A tool is just a Python function with:
- the `@tool` decorator
- type hints (so the model knows what arguments to send)
- a docstring (so the model knows *when* to use it)

In [5]:
from datetime import datetime
from zoneinfo import ZoneInfo

from strands import tool


@tool
def current_time(timezone: str = "Asia/Kolkata") -> str:
    """Get the current date and time in a given IANA timezone.

    Args:
        timezone: IANA timezone name, for example "Asia/Kolkata" or "America/New_York".
    """
    return datetime.now(ZoneInfo(timezone)).strftime("%A, %d %B %Y %I:%M %p %Z")


agent = Agent(model=model, tools=[current_time])
agent("What time is it right now in Bangalore and in New York?")

I'll check the current time in both cities simultaneously!
Tool #1: current_time

Tool #2: current_time
Here's the current time in both cities:

- 🇮🇳 **Bangalore (IST):** Sunday, 25 October 2026 — 12:45 PM
- 🇺🇸 **New York (EDT):** Sunday, 25 October 2026 — 3:15 AM

Bangalore is currently **9 hours and 30 minutes ahead** of New York.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's the current time in both cities:\n\n- 🇮🇳 **Bangalore (IST):** Sunday, 25 October 2026 — 12:45 PM\n- 🇺🇸 **New York (EDT):** Sunday, 25 October 2026 — 3:15 AM\n\nBangalore is currently **9 hours and 30 minutes ahead** of New York."}], 'metadata': {'usage': {'inputTokens': 820, 'outputTokens': 101, 'totalTokens': 921}, 'metrics': {'latencyMs': 1312, 'timeToFirstByteMs': 880}}, 'tracking_id': '729d3e7b-e326-40db-9644-df447c786adf'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'current_time': ToolMetrics(tool={'toolUseId': 'tooluse_lzM5KM3f2O46XtHFYWzRDy', 'name': 'current_time', 'input': {'timezone': 'America/New_York'}}, call_count=2, success_count=2, error_count=0, total_time=0.002537167998525547)}, cycle_durations=[1.3370697498321533, 1.5610778331756592], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='f1099677-cf5f-424e-ae68-6160dfc31946', usage=

## Step 5: Use a prebuilt tool

`strands-agents-tools` ships ready-made tools. Here the agent gets a calculator so it does not have to do arithmetic "in its head".

In [6]:
from strands_tools import calculator

agent = Agent(model=model, tools=[calculator, current_time])
agent("What is 18% GST on a laptop that costs Rs 84,999? Give me the final price too.")


Tool #1: calculator


DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.
DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 84999 * 18 / 100    │                                                                            │
│  │ Result    │ 15299.82            │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 84999 * 1.18        │                                                                            │
│  │ Result    │ 100298.82           │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here's the **GST breakdown** for your laptop 🧾💻:

| Description | Amount |
|---|---|
| 💰 Laptop Base Price | ₹84,999.00 |
| 📊 GST @ 18% | ₹15,299.82 |
| ✅ **Final Price (incl. GST)** | **₹1,00,298.82** |

### Summary:
- **GST Amount** = ₹84,999 × 18% = **₹15,299.82**
- **Final Price** = ₹84,999 + ₹15,299.82 = **₹1,00,298.82**

So, the laptop will cost you **₹1,00,298.82** after adding 18% GST. 🛒

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's the **GST breakdown** for your laptop 🧾💻:\n\n| Description | Amount |\n|---|---|\n| 💰 Laptop Base Price | ₹84,999.00 |\n| 📊 GST @ 18% | ₹15,299.82 |\n| ✅ **Final Price (incl. GST)** | **₹1,00,298.82** |\n\n### Summary:\n- **GST Amount** = ₹84,999 × 18% = **₹15,299.82**\n- **Final Price** = ₹84,999 + ₹15,299.82 = **₹1,00,298.82**\n\nSo, the laptop will cost you **₹1,00,298.82** after adding 18% GST. 🛒"}], 'metadata': {'usage': {'inputTokens': 2216, 'outputTokens': 205, 'totalTokens': 2421}, 'metrics': {'latencyMs': 2394, 'timeToFirstByteMs': 932}}, 'tracking_id': '797320be-880d-4666-95b8-096026fab94c'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_KYupQqbwEUWCNewkpVWh81', 'name': 'calculator', 'input': {'expression': '84999 * 1.18', 'mode': 'evaluate'}}, call_count=2, success_count=2, error_count=0, total_time=0.010634000000209198)

## Step 6: Shape behaviour with a system prompt

The system prompt sets the agent's role, tone, and rules for every turn.

In [7]:
agent = Agent(
    model=model,
    system_prompt=(
        "You are a patient tutor for engineers who are new to AI agents. "
        "Explain ideas with one real-world analogy and keep answers under 120 words."
    ),
)
agent("What is the difference between an LLM and an AI agent?")

## LLM vs. AI Agent

**Analogy:** An LLM is like a brilliant advisor sitting at a desk — you ask a question, they give a great answer, then they're done. An AI agent is like hiring that same advisor as an *employee* — they can take your goal, make decisions, use tools (email, spreadsheets, web searches), and complete multi-step tasks independently.

**Simply put:**

- **LLM** = thinks and responds (one turn)
- **AI Agent** = thinks, acts, observes results, and keeps going until the job is done

The LLM is often the *brain inside* the agent, but the agent adds the ability to **plan, use tools, and take action** in the real world.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "## LLM vs. AI Agent\n\n**Analogy:** An LLM is like a brilliant advisor sitting at a desk — you ask a question, they give a great answer, then they're done. An AI agent is like hiring that same advisor as an *employee* — they can take your goal, make decisions, use tools (email, spreadsheets, web searches), and complete multi-step tasks independently.\n\n**Simply put:**\n\n- **LLM** = thinks and responds (one turn)\n- **AI Agent** = thinks, acts, observes results, and keeps going until the job is done\n\nThe LLM is often the *brain inside* the agent, but the agent adds the ability to **plan, use tools, and take action** in the real world."}], 'metadata': {'usage': {'inputTokens': 56, 'outputTokens': 171, 'totalTokens': 227}, 'metrics': {'latencyMs': 3010, 'timeToFirstByteMs': 958}}, 'tracking_id': 'e2a90615-226e-4478-b6ca-60dcb55ca000'}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_

## What you just did

- Built an agent with one line of code
- Added your own tool and a prebuilt tool
- Controlled behaviour with a system prompt

Next, run the two scripts in this folder to see the same idea as plain Python files, including one built with LangGraph:

```bash
python hello_agent.py
python hello_langgraph_agent.py
```